# Big Data Analysis of ICLR Submissions (PySpark on Databricks)

**Author:** Cailey Coghlan · MSc Data Science, University of Sheffield · Module: Big Data Analytics

Analysis of **55,906 ICLR submissions (2017–2026)** joined to an **OpenAlex** citation and authorship scrape, answering six questions for the ICLR Programme Committee. Two of the extensions use Spark MLlib models: a random forest that predicts acceptance, and linear regression to test for score inflation.

**How to run:** import this notebook into Databricks, upload `iclr26v1.parquet` and `openalex.json` (unzipped) from the repository's `data` folder to a Volume, set `DATA_DIR` in the first code cell, then click **Run all**. See the repository [README](../README.md) for details.

| Cell | Content |
|---|---|
| 1 | Data loading |
| 2 | Data cleaning and quality checks (both datasets) |
| 3–4 | Q1: decision counts · extension: random forest acceptance model |
| 5–6 | Q2: accepted papers with a score below 5 · extension: reviewer disagreement |
| 7–8 | Q3: top 10 papers by mean score · extension: score-inflation regression |
| 9–10 | Q4: keyword spikes · extension: acceptance before and after the spike |
| 11–12 | Q5: OpenAlex coverage · extension: geographic analysis |
| 13–14 | Q6: low-scoring, highly cited papers · extension: accepted vs rejected citations |


In [0]:
# ============================================================
# DATA LOADING
# ============================================================

import os
from pyspark.sql import functions as F, Window

# MLlib — Feature Engineering
from pyspark.ml.feature import VectorAssembler, StandardScaler

# MLlib — Models
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression

# MLlib — Evaluation
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator
)

# MLlib — Tuning
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Calculating Pearson Correlation for Question 1 extension
from pyspark.ml.stat import Correlation

# ── DATA LOCATION ───────────────────────────────────────────────────────────
# The only setting to change if you run this notebook yourself: point DATA_DIR
# at the Databricks Volume (or folder) holding both data files.
DATA_DIR      = "/Volumes/workspace/default/ijc427/"
ICLR_FILE     = DATA_DIR + "iclr26v1.parquet"   # ICLR submissions (berenslab/iclr-dataset)
OPENALEX_FILE = DATA_DIR + "openalex.json"      # OpenAlex scrape

# Verify both data files are accessible in the volume
display(dbutils.fs.ls(DATA_DIR))

# Load the ICLR submissions dataset from parquet
ICLRdata = spark.read.parquet(ICLR_FILE)

# Load the OpenAlex dataset. A single JSON array (not JSON lines),
# so multiLine=True is required for Spark to parse the whole file correctly
openalex_raw = spark.read \
    .option("multiLine", "true") \
    .json(OPENALEX_FILE)

# Required for CrossValidator checkpointing on Databricks serverless
os.environ['SPARKML_TEMP_DFS_PATH'] = DATA_DIR

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ijc427/iclr24v2.parquet,iclr24v2.parquet,19656199,1774370026000
dbfs:/Volumes/workspace/default/ijc427/iclr25v2.parquet,iclr25v2.parquet,31710045,1774370032000
dbfs:/Volumes/workspace/default/ijc427/iclr26v1.parquet,iclr26v1.parquet,48993373,1774370037000
dbfs:/Volumes/workspace/default/ijc427/openalex.json,openalex.json,31234421,1774370253000


In [0]:
# ============================================================
# DATA CLEANING
# ============================================================

# ── ICLR DATASET ─────────────────────────────────────────────────────────────

# Preview schema and row count to confirm correct loading
print("ICLR dataset preview:")
ICLRdata.show()
iclr_total_rows = ICLRdata.count()
print("Row count:", iclr_total_rows)
print("\nICLR dataset schema:")
ICLRdata.printSchema()

# Count null values across all columns simultaneously
print("\nNull value counts per column:")
ICLRdata.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ICLRdata.columns
]).show()

# No duplicate IDs were found so no duplicate papers
duplicate_ids = ICLRdata.groupBy("id").count().filter(F.col("count") > 1).count()
print(f"Duplicate IDs: {duplicate_ids}")

# Confirm year range is complete with no unexpected values
iclr_null_years = ICLRdata.filter(F.col("year").isNull()).count()
print("Null years:", iclr_null_years)
print("\nDistinct years in dataset:")
ICLRdata.select("year").distinct().orderBy("year").show()
print("\nSubmission counts by year:")
ICLRdata.groupBy("year").count().orderBy("count", ascending=False).show(truncate=False)

# Audit decision strings to expose casing inconsistencies and empty string rows
iclr_empty_decisions = ICLRdata.filter(F.col("decision") == "").count()
print("Empty decision strings:", iclr_empty_decisions)
print("\nDecision string counts (all values):")
ICLRdata.groupBy("decision").count().orderBy("count", ascending=False).show(truncate=False)

# Check whether empty decision strings are concentrated in 2026 submissions
print("\nDecision breakdown for 2026 submissions:")
ICLRdata.filter(F.col("year") == 2026).groupBy("decision").count().orderBy("count", ascending=False).show(truncate=False)

print("\nYear breakdown for papers with empty decision strings:")
ICLRdata.filter(F.col("decision") == "").groupBy("year").count().orderBy("year").show(truncate=False)

# Check scores array for nulls, empty arrays, zero values and intra array nulls
iclr_null_scores  = ICLRdata.filter(F.col("scores").isNull()).count()
iclr_empty_scores = ICLRdata.filter(F.size(F.col("scores")) == 0).count()
iclr_zero_scores  = ICLRdata.filter(F.array_contains(F.col("scores"), 0)).count()
iclr_null_in_arr  = ICLRdata.filter(F.exists(F.col("scores"), lambda x: x.isNull())).count()
print("Null score arrays:",       iclr_null_scores)
print("Empty score arrays:",      iclr_empty_scores)
print("Zero scores:",       iclr_zero_scores)
print("Null within array:", iclr_null_in_arr)

# Check keywords array for nulls and empty arrays. Used in Q4 to find spiking topics
iclr_null_keywords  = ICLRdata.filter(F.col("keywords").isNull()).count()
iclr_empty_keywords = ICLRdata.filter(F.size(F.col("keywords")) == 0).count()
print(f"Null keywords arrays: {iclr_null_keywords}")
print(f"Empty keywords arrays: {iclr_empty_keywords}")

# Confirm empty score arrays are systematic (withdrawn/desk-rejected), not random gaps
print("\nDecision breakdown for papers with empty score arrays:")
ICLRdata.filter(
    F.size(F.col("scores")) == 0
).groupBy("decision").count().orderBy("count", ascending=False).show(truncate=False)

# Remove papers with no recorded decision or no reviewer scores
ICLRdata_clean = ICLRdata.filter(
    (F.col("decision") != "") &
    (F.size(F.col("scores")) > 0)
)
iclr_clean_rows = ICLRdata_clean.count()
print("Cleaned row count:", iclr_clean_rows)

# ── ICLR SUMMARY ─────────────────────────────────────────────────────────────

print("=" * 55)
print("ICLR DATA QUALITY SUMMARY")
print("=" * 55)
print(f"Duplicate IDs:            {duplicate_ids}")
print(f"Total rows loaded:        {iclr_total_rows}")
print(f"Null years:               {iclr_null_years}")
print(f"Empty decision strings:   {iclr_empty_decisions}")
print(f"Null scores arrays:       {iclr_null_scores}")
print(f"Empty scores arrays:      {iclr_empty_scores}")
print(f"Zero scores:              {iclr_zero_scores}")
print(f"Null elements in scores arrays:  {iclr_null_in_arr}")
print(f"Null keywords arrays:     {iclr_null_keywords}")
print(f"Empty keywords arrays:    {iclr_empty_keywords}")
print(f"Rows retained after clean:{iclr_clean_rows}")
print(f"Rows removed:             {iclr_total_rows - iclr_clean_rows}")
print("=" * 55)

# ── OPENALEX DATASET ──────────────────────────────────────────────────────────
# openalex.json is a single JSON array, not JSON lines
# multiLine=True is required so Spark reads the whole file as one document
# rather than attempting to parse each line independently.

print("\nOpenAlex dataset preview:")
openalex_raw.show()

# Inspecting the schema for the variables of interest.
print("\nOpenAlex schema for key fields:")
openalex_raw.select("title", "contains_iclr", "authorships", "cited_by_count").printSchema()

# Identifying the span of the publication years in the dataset
print("\nOpenAlex publication year range:")
openalex_raw.agg(
    F.min("publication_year").alias("earliest"),
    F.max("publication_year").alias("latest")
).show()

# Null and empty string check on title: used as the join key throughout Q5 and Q6
print("\nNull counts for title and authorships:")
openalex_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["title", "authorships"]
]).show()
empty_titles = openalex_raw.filter(F.trim(F.col("title")) == "").count()
print(f"Empty or whitespace-only titles: {empty_titles}")

# # Check for duplicate titles after normalising to lowercase and removing excess whitespace.
oa_title_counts = openalex_raw \
    .select(F.trim(F.lower(F.col("title"))).alias("norm_title")) \
    .groupBy("norm_title") \
    .count()
oa_duplicate_titles = oa_title_counts.filter(F.col("count") > 1).count()
print(f"Duplicate normalised titles in OpenAlex: {oa_duplicate_titles}")

# Check authorships array for null and empty entries at the top level
null_authorships  = openalex_raw.filter(F.col("authorships").isNull()).count()
empty_authorships = openalex_raw.filter(F.size(F.col("authorships")) == 0).count()
print(f"Null authorships arrays: {null_authorships}")
print(f"Empty authorships arrays: {empty_authorships}")

# Check coverage of the nested countries array: used in Q5 extension for
# geographic analysis of submission volume, acceptance rate, and reviewer scores
authorships_exploded = openalex_raw.select(F.explode("authorships").alias("authorship"))
empty_countries = authorships_exploded.filter(
    F.size(F.col("authorship.countries")) == 0
).count()
print("Authorships with empty countries array:", empty_countries)

# Check citation_normalized_percentile.value for nulls and zeros: Q6 uses this
# field to identify papers in the 95th citation percentile
oa_null_percentile = openalex_raw.filter(
    F.col("citation_normalized_percentile.value").isNull()
).count()
oa_zero_percentile = openalex_raw.filter(
    F.col("citation_normalized_percentile.value") == 0
).count()
print(f"Null citation_normalized_percentile.value: {oa_null_percentile}")
print(f"Zero citation_normalized_percentile.value: {oa_zero_percentile}")

# Check publication_year is populated: confirms the field is usable for year based analysis
print("\nDistinct publication years in OpenAlex:")
openalex_raw.select("publication_year").distinct().orderBy("publication_year").show()

# ── OPENALEX SUMMARY ──────────────────────────────────────────────────────────

print("=" * 55)
print("OPENALEX DATA QUALITY SUMMARY")
print("=" * 55)
total_records = openalex_raw.count()
print(f"Total records loaded:        {total_records}")
print(f"Duplicate normalised titles: {oa_duplicate_titles}")
print(f"Empty/null titles:           {empty_titles}")
print(f"Null authorships arrays:     {null_authorships}")
print(f"Empty authorships arrays:    {empty_authorships}")
print(f"Authorships empty countries: {empty_countries}")
print("=" * 55)
print(f"Null percentile values:      {oa_null_percentile}")
print(f"Zero percentile values:      {oa_zero_percentile}")

ICLR dataset preview:
+----+---------+--------------------+--------------------+--------------------+----------+--------------------+------------+--------------------+--------------------+
|year|       id|               title|            abstract|             authors|author_ids|            decision|      scores|            keywords|              labels|
+----+---------+--------------------+--------------------+--------------------+----------+--------------------+------------+--------------------+--------------------+
|2017|B1-Hhnslg|Prototypical Netw...|A recent approach...|Jake Snell, Kevin...|          |              Reject|   [6, 4, 5]|[deep learning, t...|   transfer learning|
|2017|B1-q5Pqxl|Machine Comprehen...|Machine comprehen...|Shuohang Wang, Ji...|          |     Accept (Poster)|   [6, 6, 7]|[natural language...|     language models|
|2017|B16Jem9xe|Learning in Impli...|Generative advers...|Shakir Mohamed, B...|          |Invite to Worksho...|   [8, 7, 6]|[unsupervised lea..

In [0]:
# ============================================================
# QUESTION 1 — ACCEPTED, REJECTED AND WITHDRAWN PAPER COUNTS
# ============================================================
# Uses ICLRdata (not ICLRdata_clean) — decisions are needed but scores are not.


ICLR_q1 = ICLRdata.filter(F.col("decision") != "") \
    .withColumn("category",
        F.when(F.col("decision").rlike(
            "(?i)accept|oral|poster|spotlight|workshop|talk|notable"), "Accepted")
         .when(F.col("decision").rlike(
            "(?i)withdraw"), "Withdrawn")
         .otherwise("Rejected")
    )

# Count papers in each category
ICLR_q1.groupBy("category") \
     .count() \
     .orderBy("count", ascending=False) \
     .show()


+---------+-----+
| category|count|
+---------+-----+
| Rejected|17290|
| Accepted|11349|
|Withdrawn| 7594|
+---------+-----+



In [0]:

# ============================================================
# QUESTION 1 EXTENSION — PREDICTING ACCEPTANCE USING
# REVIEWER SCORES WITH RANDOM FOREST
# ============================================================
# Binary classification using Random Forest.
# Features are derived from the scores array; labels are 1=Accepted, 0=Not Accepted.
# Stratified 80/20 split preserves class proportions across train and test.
# CrossValidator tunes tree depth and number of trees via 3-fold cross-validation.

# Derive five score based features from the scores array
ICLRdata_features = ICLRdata_clean \
    .withColumn("mean_score",
        (F.aggregate(F.col("scores"), F.lit(0.0), lambda acc, x: acc + x.cast("double")) /
         F.size(F.col("scores")))
    ) \
    .withColumn("min_score",   F.array_min(F.col("scores")).cast("double")) \
    .withColumn("max_score",   F.array_max(F.col("scores")).cast("double")) \
    .withColumn("num_reviewers", F.size(F.col("scores"))) \
    .withColumn("std_score",
        F.sqrt(
            F.aggregate(
                F.col("scores"), F.lit(0.0),
                lambda acc, x: acc + (x.cast("double") - F.col("mean_score")) ** 2
            ) / F.size(F.col("scores"))
        )
    )

# Show first 5 rows with full descriptive column names
ICLRdata_features.select(
    F.col("mean_score").alias("Mean Score"),
    F.col("min_score").alias("Minimum Score"),
    F.col("max_score").alias("Maximum Score"),
    F.col("num_reviewers").alias("Number of Reviewers"),
    F.col("std_score").alias("Standard Deviation of Scores (Reviewer Disagreement)")
).show(5)

# Add binary label: 1.0 = Accepted, 0.0 = Not Accepted
# Withdrawn papers are treated as Not Accepted. They represent unsuccessful
# submissions regardless of the reason for withdrawal
ICLRdata_labelled = ICLRdata_features.withColumn("label",
    F.when(F.col("decision").rlike(
        "(?i)accept|oral|poster|spotlight|workshop|talk|notable"), 1.0)
     .otherwise(0.0)
)

feature_names  = ["mean_score", "min_score", "max_score", "num_reviewers", "std_score"]
feature_labels = ["Mean Score", "Minimum Score", "Maximum Score", "Number of Reviewers", "Standard Deviation of Scores (Reviewer Disagreement)"]

# Assemble feature columns into a single vector required by PySpark ML
assembler = VectorAssembler(inputCols=feature_names, outputCol="features")
ICLRdata_assembled = assembler.transform(ICLRdata_labelled)

# Testing for multicollinearity
pearson_matrix = Correlation.corr(ICLRdata_assembled, "features", "pearson").collect()[0][0]

print("Pearson Correlation Matrix:")
print(f"{'':>15}", end="")
for name in feature_names:
    print(f"{name:>15}", end="")
print()
for i, row_name in enumerate(feature_names):
    print(f"{row_name:>15}", end="")
    for j in range(len(feature_names)):
        print(f"{pearson_matrix[i, j]:>15.3f}", end="")
    print()

# Stratified split: each class split separately to preserve class proportions
class_not_accepted = ICLRdata_assembled.filter(F.col("label") == 0.0)
class_accepted     = ICLRdata_assembled.filter(F.col("label") == 1.0)
train_not, test_not = class_not_accepted.randomSplit([0.8, 0.2], seed=42)
train_acc, test_acc = class_accepted.randomSplit([0.8, 0.2], seed=42)
train = train_not.union(train_acc)
test  = test_not.union(test_acc)

print("\nTraining set class distribution:")
train.groupBy("label").count().orderBy("label").show()
print("Test set class distribution:")
test.groupBy("label").count().orderBy("label").show()

binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

# Train Random Forest with feature importances to show which score statistics
# drive acceptance decisions. CrossValidator tunes maxDepth and numTrees to balance
# expressivity against overfitting
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=200, maxDepth=10, minInstancesPerNode=5,
    featureSubsetStrategy="sqrt", seed=42
)
rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [5, 10]) \
    .addGrid(rf.numTrees, [100]) \
    .build()
rf_cv = CrossValidator(
    estimator=rf, estimatorParamMaps=rf_paramGrid,
    evaluator=binary_evaluator, numFolds=3, seed=42, parallelism=4
)
print("\nTraining Random Forest with Cross Validation...")
rf_cv_model   = rf_cv.fit(train)
rf_best_model = rf_cv_model.bestModel
print("Best maxDepth:", rf_best_model.getOrDefault("maxDepth"))
print("Best numTrees:", rf_best_model.getOrDefault("numTrees"))

# Print feature importances to show the relative contribution of each score feature
print("\nRandom Forest Feature Importances:")
for label, importance in zip(feature_labels, rf_best_model.featureImportances):
    bar = "█" * int(importance * 50)
    print(f"{label:<52}: {importance:.4f} {bar}")

# Generate predictions on the held-out test set
rf_predictions = rf_best_model.transform(test)

# Compute accuracy and AUC
rf_accuracy = rf_predictions.filter(
    F.col("prediction") == F.col("label")
).count() / rf_predictions.count()

rf_auc = binary_evaluator.evaluate(rf_predictions)

# Print confusion matrix
print("\nRandom Forest Confusion Matrix:")
print(f"{'':>25} {'Predicted: Not Accepted':>25} {'Predicted: Accepted':>20}")
cm = rf_predictions.groupBy("label") \
    .pivot("prediction", [0.0, 1.0]) \
    .count() \
    .fillna(0) \
    .orderBy("label")
class_names = ["Actual: Not Accepted", "Actual: Accepted"]
for row in cm.collect():
    label  = int(row["label"])
    pred_0 = int(row["0.0"] or 0)
    pred_1 = int(row["1.0"] or 0)
    print(f"{class_names[label]:>25} {pred_0:>25} {pred_1:>20}")

# Print final evaluation summary
print("\n" + "=" * 45)
print("RANDOM FOREST EVALUATION RESULTS")
print("=" * 45)
print(f"{'Metric':<25} {'Random Forest':>15}")
print("-" * 45)
print(f"{'Accuracy':<25} {rf_accuracy:>15.4f}")
print(f"{'AUC':<25} {rf_auc:>15.4f}")
print("=" * 45)
 

+-----------------+-------------+-------------+-------------------+----------------------------------------------------+
|       Mean Score|Minimum Score|Maximum Score|Number of Reviewers|Standard Deviation of Scores (Reviewer Disagreement)|
+-----------------+-------------+-------------+-------------------+----------------------------------------------------+
|              5.0|          4.0|          6.0|                  3|                                   0.816496580927726|
|6.333333333333333|          6.0|          7.0|                  3|                                  0.4714045207910317|
|              7.0|          6.0|          8.0|                  3|                                   0.816496580927726|
|5.666666666666667|          5.0|          6.0|                  3|                                  0.4714045207910317|
|              7.0|          5.0|          9.0|                  3|                                   1.632993161855452|
+-----------------+-------------

In [0]:
# ========================================================================
# QUESTION 2 - NUMBER OF PAPERS ACCEPTED WITH AT LEAST ONE SCORE UNDER 5
# ========================================================================

# Filter to accepted papers only using the same regex as Q1
df_accepted = ICLRdata_clean.filter(
    F.col("decision").rlike("(?i)accept|oral|poster|spotlight|workshop|talk|notable")
)

# Count accepted papers where at least one score in the array is below 5
q2_count = df_accepted.filter(
    F.exists(F.col("scores"), lambda x: x < 5)
).count()

total_accepted = df_accepted.count()
print(f"Total accepted papers: {total_accepted}")
print(f"Q2 Answer: {q2_count} accepted papers had at least one score under 5")


Total accepted papers: 11349
Q2 Answer: 1596 accepted papers had at least one score under 5


In [0]:

#======================================================================
# QUESTION 2 EXTENSION: REVIEWER DISAGREEMENT ANALYSIS 
#=======================================================================
# ── COMPUTE SCORE STANDARD DEVIATION PER PAPER ───────────────────────────────
# mean_score is needed first so std dev can reference it in the same aggregate
df_with_stats = ICLRdata_clean \
    .withColumn("mean_score",
        F.aggregate(F.col("scores"), F.lit(0.0),
            lambda acc, x: acc + x.cast("double")
        ) / F.size(F.col("scores"))
    ) \
    .withColumn("score_std",
        F.sqrt(
            F.aggregate(F.col("scores"), F.lit(0.0),
                lambda acc, x: acc + (x.cast("double") - F.col("mean_score")) ** 2
            ) / F.size(F.col("scores"))
        )
    ) \
    .withColumn("accepted",
        F.when(F.col("decision").rlike(
            "(?i)accept|oral|poster|spotlight|workshop|talk|notable"), 1.0
        ).otherwise(0.0)
    )

# ── BIN INTO DISAGREEMENT TIERS ───────────────────────────────────────────────
# Use approxQuantile to find the 33rd and 66th percentile thresholds
p33, p66 = df_with_stats.approxQuantile("score_std", [0.33, 0.66], 0.001)
print(f"Disagreement thresholds — Low: Score Standard Deviation < {p33:.3f}, Medium: Score Standard Deviation {p33:.3f}–{p66:.3f}, High: Score Standard Deviation > {p66:.3f}")

df_tiered = df_with_stats.withColumn("disagreement_tier",
    F.when(F.col("score_std") <= p33, "Low")
     .when(F.col("score_std") <= p66, "Medium")
     .otherwise("High")
)

# ── ACCEPTANCE RATE BY DISAGREEMENT TIER ─────────────────────────────────────
print("\nAcceptance rate by reviewer disagreement tier:")
df_tiered \
    .groupBy("disagreement_tier") \
    .agg(
        F.count("id").alias("Total Paper Count in Tier"),
        F.round(F.avg("score_std"), 3).alias("Score Standard Deviation in Tier"),
        F.round(F.avg("accepted") * 100, 1).alias("Acceptance Rate in Tier")
    ) \
    .orderBy("disagreement_tier") \
    .toDF("Disagreement Tier", "Total Paper Count in Tier", "Score Standard Deviation in Tier", "Acceptance Rate in Tier (%)") \
    .show(truncate=False)

# ── DISAGREEMENT TIER FOR PAPERS WITH AT LEAST ONE LOW SCORE ─────────────────
# Cross tabulate the Q2 result with disagreement tier. Do papers with a low
# score tend to come from high disagreement reviews or low disagreement ones?
print("\nAccepted papers by reviewer disagreement tier and whether they contain a low score:")
df_tiered \
    .filter(F.col("accepted") == 1.0) \
    .withColumn("has_low_score",
        F.when(F.exists(F.col("scores"), lambda x: x < 5), "Yes").otherwise("No")
    ) \
    .groupBy("disagreement_tier", "has_low_score") \
    .agg(F.count("id").alias("paper_count")) \
    .orderBy("disagreement_tier", "has_low_score") \
    .toDF("Disagreement Tier", "Has at Least One Low Score", "Number of Accepted Papers") \
    .show(truncate=False)

# ── DISAGREEMENT TREND OVER TIME ──────────────────────────────────────────────
# Score standard deviation and Average score of accepted papers per year
print("\nReviewer disagreement (score standard deviation) and average score on accepted papers by year:")
df_tiered \
    .filter(F.col("accepted") == 1.0) \
    .groupBy("year") \
    .agg(
        F.count("id").alias("accepted_papers"),
        F.round(F.avg("score_std"), 3).alias("avg_score_std"),
        F.round(F.avg("mean_score"), 3).alias("avg_mean_score")
    ) \
    .orderBy("year") \
    .toDF("Year", "Number of Accepted Papers", "Score Standard Deviation for Accepted Papers (Reviewer Disagreement)", "Average Score for Accepted Papers") \
    .show(truncate=False)

Disagreement thresholds — Low: Score Standard Deviation < 0.866, Medium: Score Standard Deviation 0.866–1.225, High: Score Standard Deviation > 1.225

Acceptance rate by reviewer disagreement tier:
+-----------------+-------------------------+--------------------------------+---------------------------+
|Disagreement Tier|Total Paper Count in Tier|Score Standard Deviation in Tier|Acceptance Rate in Tier (%)|
+-----------------+-------------------------+--------------------------------+---------------------------+
|High             |11334                    |1.628                           |25.4                       |
|Low              |14155                    |0.476                           |35.5                       |
|Medium           |10128                    |1.04                            |34.1                       |
+-----------------+-------------------------+--------------------------------+---------------------------+


Accepted papers by reviewer disagreement tier and w

In [0]:
# ============================================================
# QUESTION 3 — TOP 10 PAPERS BY HIGHEST MEAN SCORE
# ============================================================
# Uses ICLRdata filtered to non empty scores only. No decision required.
# This avoids excluding valid submissions that lack a recorded outcome.

# Compute mean score per paper from the scores array
df_q3_base = ICLRdata.filter(F.size(F.col("scores")) > 0) \
    .withColumn("mean_score",
        F.aggregate(
            F.col("scores"), F.lit(0.0),
            lambda acc, x: acc + x.cast("double")
        ) / F.size(F.col("scores"))
    )

# Select the top 10 papers by mean score
df_q3 = df_q3_base \
    .select("id", "mean_score", "year") \
    .orderBy(F.col("mean_score").desc(), F.col("year").desc(), F.col("id").asc()) \
    .limit(10)

print("Q3 Answer — Top 10 papers by highest mean score:")
df_q3.show(truncate=False)



Q3 Answer — Top 10 papers by highest mean score:
+----------+-----------------+----+
|id        |mean_score       |year|
+----------+-----------------+----+
|u1cQYxRI1H|10.0             |2025|
|Sy8gdB9xx |9.666666666666666|2017|
|6Mxhg9PtDE|9.5              |2025|
|88nT0j5jAn|9.333333333333334|2023|
|LyJi5ugyJx|9.2              |2025|
|DJSZGGZYVi|9.0              |2025|
|Ha6RTeWMd0|9.0              |2025|
|WCRQFlji2q|9.0              |2025|
|YrycTjllL0|9.0              |2025|
|gc8QAQfXv6|9.0              |2025|
+----------+-----------------+----+



In [0]:
# ============================================================
# QUESTION 3 EXTENSION — SCORE INFLATION ANALYSIS USING
# LINEAR REGRESSION
# ============================================================

# Prepare three subsets: all papers, accepted only, rejected only
df_all          = df_q3_base.select("year", "mean_score")
df_q3_known     = df_q3_base.filter(F.col("decision") != "")
df_accepted_reg = df_q3_known \
    .filter(F.col("decision").rlike(
        "(?i)accept|oral|poster|spotlight|workshop|talk|notable")) \
    .select("year", "mean_score")
df_rejected_reg = df_q3_known \
    .filter(~F.col("decision").rlike(
        "(?i)accept|oral|poster|spotlight|workshop|talk|notable")) \
    .select("year", "mean_score")

print(f"All papers:      {df_all.count()} rows")
print(f"Accepted papers: {df_accepted_reg.count()} rows")
print(f"Rejected papers: {df_rejected_reg.count()} rows")

# Wrap year into a feature vector and apply an 80/20 train/test split to each subset
assembler_reg = VectorAssembler(inputCols=["year"], outputCol="features")
df_all_assembled      = assembler_reg.transform(df_all)
df_accepted_assembled = assembler_reg.transform(df_accepted_reg)
df_rejected_assembled = assembler_reg.transform(df_rejected_reg)

train_all,      test_all      = df_all_assembled.randomSplit([0.8, 0.2], seed=42)
train_accepted, test_accepted = df_accepted_assembled.randomSplit([0.8, 0.2], seed=42)
train_rejected, test_rejected = df_rejected_assembled.randomSplit([0.8, 0.2], seed=42)

print("\nTrain/test split sizes:")
print(f"All papers      — Train: {train_all.count()}, Test: {test_all.count()}")
print(f"Accepted papers — Train: {train_accepted.count()}, Test: {test_accepted.count()}")
print(f"Rejected papers — Train: {train_rejected.count()}, Test: {test_rejected.count()}")

# Fit a separate linear regression on each training set
lr_reg = LinearRegression(featuresCol="features", labelCol="mean_score")
lr_model_all      = lr_reg.fit(train_all)
lr_model_accepted = lr_reg.fit(train_accepted)
lr_model_rejected = lr_reg.fit(train_rejected)

# Print slope and intercept for each group. Slope shows score change per year
print("\n" + "=" * 58)
print("LINEAR REGRESSION COEFFICIENTS")
print("=" * 58)
print(f"{'Group':<20} {'Slope (per year)':>18} {'Intercept':>15}")
print("-" * 58)
print(f"{'All papers':<20} "
      f"{lr_model_all.coefficients[0]:>18.4f} "
      f"{lr_model_all.intercept:>15.4f}")
print(f"{'Accepted papers':<20} "
      f"{lr_model_accepted.coefficients[0]:>18.4f} "
      f"{lr_model_accepted.intercept:>15.4f}")
print(f"{'Rejected papers':<20} "
      f"{lr_model_rejected.coefficients[0]:>18.4f} "
      f"{lr_model_rejected.intercept:>15.4f}")
print("=" * 58)

# Generate predictions on each held out test set
predictions_all      = lr_model_all.transform(test_all)
predictions_accepted = lr_model_accepted.transform(test_accepted)
predictions_rejected = lr_model_rejected.transform(test_rejected)

# Evaluate each model with R² as the primary metric
r2_evaluator = RegressionEvaluator(
    labelCol="mean_score", predictionCol="prediction", metricName="r2"
)

r2_all      = r2_evaluator.evaluate(predictions_all)
r2_accepted = r2_evaluator.evaluate(predictions_accepted)
r2_rejected = r2_evaluator.evaluate(predictions_rejected)

# Print R² results for each group
print("\n" + "=" * 58)
print("MODEL EVALUATION RESULTS (TEST SET)")
print("=" * 58)
print(f"{'Group':<20} {'R²':>18}")
print("-" * 58)
print(f"{'All papers':<20} {r2_all:>18.4f}")
print(f"{'Accepted papers':<20} {r2_accepted:>18.4f}")
print(f"{'Rejected papers':<20} {r2_rejected:>18.4f}")
print("=" * 58)

All papers:      35617 rows
Accepted papers: 11349 rows
Rejected papers: 24268 rows

Train/test split sizes:
All papers      — Train: 28507, Test: 7110
Accepted papers — Train: 9151, Test: 2198
Rejected papers — Train: 19438, Test: 4830

LINEAR REGRESSION COEFFICIENTS
Group                  Slope (per year)       Intercept
----------------------------------------------------------
All papers                       0.0128        -20.8712
Accepted papers                 -0.0003          7.1638
Rejected papers                  0.0321        -60.4755

MODEL EVALUATION RESULTS (TEST SET)
Group                                R²
----------------------------------------------------------
All papers                       0.0006
Accepted papers                 -0.0001
Rejected papers                  0.0059


In [0]:
#======================================================
#         QUESTION 4 — KEYWORDS THAT SPIKED <5 TO >50         
#======================================================
# Uses ICLRdata (all submissions).
# Keywords normalised to lowercase with collapsed internal whitespace.

# Explode and normalise keywords. One row per keyword per paper
keywords_exploded = ICLRdata \
    .filter(F.size(F.col("keywords")) > 0) \
    .select("year", F.explode(F.col("keywords")).alias("keyword")) \
    .withColumn("keyword", F.lower(F.col("keyword"))) \
    .withColumn("keyword", F.trim(F.col("keyword"))) \
    .withColumn("keyword", F.regexp_replace(F.col("keyword"), "\\s+", " ")) \
    .filter(F.col("keyword") != "")

# Count keyword occurrences per year
kw_counts = keywords_exploded \
    .groupBy("year", "keyword") \
    .count() \
    .withColumnRenamed("count", "kw_count")

# Self join on keyword and consecutive year to find year-on-year pairs
kw_prev = kw_counts \
    .withColumnRenamed("year", "year_prev") \
    .withColumnRenamed("kw_count", "count_prev")
kw_next = kw_counts \
    .withColumnRenamed("year", "year_next") \
    .withColumnRenamed("kw_count", "count_next")

kw_joined = kw_prev.join(
    kw_next,
    on=(kw_prev["keyword"] == kw_next["keyword"]) &
       (kw_prev["year_prev"] + 1 == kw_next["year_next"])
).select(
    kw_prev["keyword"],
    "year_prev", "count_prev",
    "year_next", "count_next"
)

# Apply the spike filter: fewer than 5 one year, more than 50 the next
kw_q4 = kw_joined.filter(
    (F.col("count_prev") < 5) & (F.col("count_next") > 50)
)

print("Q4 Answer — Keywords that spiked from <5 to >50:")
kw_q4.orderBy(F.col("count_next").desc()) \
    .toDF("Keyword", "Pre-Spike Year", "Paper Count in Pre-Spike Year", "Spike Year", "Paper Count in Spike Year") \
    .show(10, truncate=False)


Q4 Answer — Keywords that spiked from <5 to >50:
+----------------------+--------------+-----------------------------+----------+-------------------------+
|Keyword               |Pre-Spike Year|Paper Count in Pre-Spike Year|Spike Year|Paper Count in Spike Year|
+----------------------+--------------+-----------------------------+----------+-------------------------+
|test-time scaling     |2025          |3                            |2026      |87                       |
|llm                   |2023          |4                            |2024      |76                       |
|large reasoning models|2025          |1                            |2026      |57                       |
|spatial reasoning     |2025          |3                            |2026      |57                       |
+----------------------+--------------+-----------------------------+----------+-------------------------+



In [0]:

#==============================================================================
# QUESTION 4 — EXTENSION: KEYWORD ACCEPTANCE RATE BEFORE AND AFTER THE SPIKE          
#==============================================================================          

# ── PREPARE SPIKING KEYWORD TABLE ────────────────────────────────────────────

# Keep only the first spike per keyword; extract pre spike and spike years
w_first_spike = Window.partitionBy("keyword").orderBy("year_next")
spiking_kws = kw_q4 \
    .withColumn("rank", F.rank().over(w_first_spike)) \
    .filter(F.col("rank") == 1) \
    .select(
        "keyword",
        F.col("year_prev").alias("pre_spike_year"),
        F.col("year_next").alias("spike_year"),
        F.col("count_prev").alias("count_pre"),
        F.col("count_next").alias("count_spike")
    )

# ── BUILD KEYWORD-PAPER ACCEPTANCE TABLE ──────────────────────────────────────

# Explode and normalise keywords. Same cleaning as Q4 core so join keys match
kw_paper_acceptance = ICLRdata \
    .filter(F.col("decision") != "") \
    .withColumn("accepted",
        F.when(F.col("decision").rlike(
            "(?i)accept|oral|poster|spotlight|workshop|talk|notable"), 1.0
        ).otherwise(0.0)
    ) \
    .filter(F.size(F.col("keywords")) > 0) \
    .select(
        "year", "accepted",
        F.explode(F.col("keywords")).alias("keyword")
    ) \
    .withColumn("keyword", F.lower(F.col("keyword"))) \
    .withColumn("keyword", F.trim(F.col("keyword"))) \
    .withColumn("keyword", F.regexp_replace(F.col("keyword"), "\\s+", " ")) \
    .filter(F.col("keyword") != "")

# ── YEAR-BY-YEAR TRAJECTORY ───────────────────────────────────────────────────

# Full year-by-year acceptance rate per spiking keyword. Reveals whether
# any shift was gradual or sudden
kw_yearly_acceptance = kw_paper_acceptance \
    .join(spiking_kws.select("keyword", "spike_year"), on="keyword", how="inner") \
    .groupBy("keyword", "spike_year", "year") \
    .agg(
        F.round(F.avg("accepted") * 100, 1).alias("acceptance_rate_percentage"),
        F.count("accepted").alias("paper_count")
    ) \
    .withColumn("period",
        F.when(F.col("year") < F.col("spike_year"),  "Pre-spike")
         .when(F.col("year") == F.col("spike_year"), "Spike year")
         .otherwise("Post-spike")
    ) \
    .orderBy("keyword", "year")

print("Year-by-year acceptance rate trajectory for each spiking keyword:")
kw_yearly_acceptance.select(
    "keyword", "year", "period", "paper_count", "acceptance_rate_percentage"
).show(50, truncate=False)

Year-by-year acceptance rate trajectory for each spiking keyword:
+----------------------+----+----------+-----------+--------------------------+
|keyword               |year|period    |paper_count|acceptance_rate_percentage|
+----------------------+----+----------+-----------+--------------------------+
|large reasoning models|2025|Pre-spike |1          |0.0                       |
|large reasoning models|2026|Spike year|1          |0.0                       |
|llm                   |2023|Pre-spike |4          |50.0                      |
|llm                   |2024|Spike year|76         |25.0                      |
|llm                   |2025|Post-spike|350        |29.7                      |
|llm                   |2026|Post-spike|6          |0.0                       |
|spatial reasoning     |2024|Pre-spike |1          |0.0                       |
|spatial reasoning     |2025|Pre-spike |3          |33.3                      |
|test-time scaling     |2025|Pre-spike |3          |66

In [0]:
# ============================================================
# QUESTION 5 — ICLR PAPERS WITH OPENALEX DATA, BY YEAR
# ============================================================
# All 2,245 OpenAlex records are used. Every record is a potential ICLR match.

# Select only the columns needed from OpenAlex
openalex_iclr = openalex_raw \
    .select(
        F.col("title").alias("oa_title"),
        F.col("authorships")
    )

# Join ICLR to OpenAlex on matching title strings
q5_core_joined = ICLRdata.join(
    openalex_iclr,
    ICLRdata.title == openalex_iclr.oa_title,
    how="inner"
)

# Count total matched ICLR papers across all years
total_matched = q5_core_joined.select(
    F.count("id")
).collect()[0][0]

print(f"Total matched papers (ICLR ∩ OpenAlex): {total_matched}")
total_iclr = ICLRdata.count()
match_rate  = round(total_matched / total_iclr * 100, 1)
print(f"Total ICLR submissions:          {total_iclr}")
print(f"Overall match rate:              {match_rate}%")

# Compute matched paper count and match rate per submission year
total_by_year = ICLRdata \
    .groupBy("year") \
    .agg(F.count("id").alias("total_submissions"))

print("Q5 Answer — matched papers and match rate by year:")
q5_core_joined \
    .groupBy("year") \
    .agg(F.count("id").alias("matched_papers")) \
    .join(total_by_year, on="year", how="left") \
    .withColumn(
        "match_rate_pct",
        F.round(
            F.col("matched_papers") / F.col("total_submissions") * 100, 1
        )
    ) \
    .orderBy("year") \
    .toDF("ICLR Submission Year", "Number of Matched Papers", "Total Number of Papers in ICLR Dataset", "Match Rate (%)") \
    .show(truncate=False)

Total matched papers (ICLR ∩ OpenAlex): 1593
Total ICLR submissions:          55906
Overall match rate:              2.8%
Q5 Answer — matched papers and match rate by year:
+--------------------+------------------------+--------------------------------------+--------------+
|ICLR Submission Year|Number of Matched Papers|Total Number of Papers in ICLR Dataset|Match Rate (%)|
+--------------------+------------------------+--------------------------------------+--------------+
|2017                |154                     |489                                   |31.5          |
|2018                |309                     |1012                                  |30.5          |
|2019                |199                     |1569                                  |12.7          |
|2020                |433                     |2593                                  |16.7          |
|2021                |423                     |3009                                  |14.1          |
|2022      

In [0]:

# ======================================================================
# QUESTION 5 EXTENSION — GEOGRAPHIC ANALYSIS OF ICLR SUBMISSIONS
# ======================================================================

# Part 1: Country submission volume
# Parts 2 & 3: Acceptance rate and average reviewer score by country,
# combined into a single output.

# ── BUILD EXTENSION JOIN ──────────────────────────────────────────────────────

# Known decisions only, binary accepted flag added, title normalised for joining
iclr_decisions_titled = ICLRdata \
    .filter(F.col("decision") != "") \
    .withColumn(
        "norm_title",
        F.trim(F.lower(F.col("title")))
    ) \
    .withColumn(
        "accepted",
        F.when(
            F.col("decision").rlike(
                "(?i)accept|oral|poster|spotlight|workshop|talk|notable"
            ),
            1.0
        ).otherwise(0.0)
    )

# Normalise OpenAlex titles for joining
openalex_iclr_norm = openalex_raw \
    .select(
        F.trim(F.lower(F.col("title"))).alias("oa_title"),
        F.col("authorships")
    )

q5_ext_joined = iclr_decisions_titled.join(
    openalex_iclr_norm,
    iclr_decisions_titled.norm_title == openalex_iclr_norm.oa_title,
    how="inner"
)

total_ext_matched = q5_ext_joined.select(
    F.countDistinct("id")
).collect()[0][0]
print(f"Matched papers with known decision: {total_ext_matched}")

# ISO 3166-1 alpha-2 country code to full name mapping
country_names = {
    "US": "United States", "GB": "United Kingdom", "CN": "China",
    "CA": "Canada", "IL": "Israel", "DE": "Germany", "CH": "Switzerland",
    "FR": "France", "KR": "South Korea", "JP": "Japan", "NL": "Netherlands",
    "SG": "Singapore", "IT": "Italy", "IN": "India", "AU": "Australia",
    "RU": "Russia", "ES": "Spain", "AT": "Austria", "TW": "Taiwan",
    "SE": "Sweden", "BE": "Belgium", "DK": "Denmark", "FI": "Finland",
    "NO": "Norway", "PT": "Portugal", "CZ": "Czech Republic",
    "PL": "Poland", "BR": "Brazil", "IE": "Ireland", "HU": "Hungary",
    "BG": "Bulgaria", "SA": "Saudi Arabia"
}

country_mapping = spark.createDataFrame(
    [(k, v) for k, v in country_names.items()],
    ["country_code", "country_name"]
)

# ── PART 1 — COUNTRY SUBMISSION VOLUME ───────────────────────────────────────

# Normalise OpenAlex titles for core join used in Part 1
openalex_iclr_norm_core = openalex_raw \
    .select(
        F.trim(F.lower(F.col("title"))).alias("oa_title"),
        F.col("authorships")
    )

# Normalise ICLR titles for core join
iclr_all_titled = ICLRdata \
    .withColumn(
        "norm_title",
        F.trim(F.lower(F.col("title")))
    )

q5_core_joined_norm = iclr_all_titled.join(
    openalex_iclr_norm_core,
    iclr_all_titled.norm_title == openalex_iclr_norm_core.oa_title,
    how="inner"
)

# Double explode authorships then countries to get one row per country per paper
q5_countries = q5_core_joined_norm \
    .select(
        "id",
        F.explode("authorships").alias("authorship")
    ) \
    .select(
        "id",
        F.explode("authorship.countries").alias("country_code")
    ) \
    .groupBy("country_code") \
    .agg(F.countDistinct("id").alias("paper_count")) \
    .join(country_mapping, on="country_code", how="left") \
    .withColumn("country_name",
        F.coalesce(F.col("country_name"), F.col("country_code"))
    ) \
    .orderBy(F.col("paper_count").desc())

print("Part 1 — Top 10 countries by matched paper count:")
q5_countries.select("country_name", "paper_count") \
    .toDF("Country Name", "Number of Papers") \
    .show(10, truncate=False)

# ── PART 2 — ACCEPTANCE RATE BY COUNTRY ──────────────────────────────────────

# Acceptance rate per country; F.avg on a binary accepted column equals the rate
# Minimum 10 papers enforced for statistical reliability
q5_country_acceptance = q5_ext_joined \
    .select(
        "id", "accepted",
        F.explode("authorships").alias("authorship")
    ) \
    .select(
        "id", "accepted",
        F.explode("authorship.countries").alias("country_code")
    ) \
    .groupBy("country_code") \
    .agg(
        F.countDistinct("id").alias("paper_count"),
        F.round(F.avg("accepted"), 3).alias("acceptance_rate")
    ) \
    .filter(F.col("paper_count") >= 10)

# ── PART 3 — AVERAGE REVIEWER SCORE BY COUNTRY ───────────────────────────────

# Computed separately from Part 2 as it requires filtering to non-empty score
# arrays, which slightly changes the paper count per country. Joined to Part 2
# to produce a single combined table.
q5_country_scores = q5_ext_joined \
    .withColumn("mean_score",
        F.aggregate(
            F.col("scores"), F.lit(0.0),
            lambda acc, x: acc + x.cast("double")
        ) / F.size(F.col("scores"))
    ) \
    .filter(F.size(F.col("scores")) > 0) \
    .select(
        "id", "mean_score",
        F.explode("authorships").alias("authorship")
    ) \
    .select(
        "id", "mean_score",
        F.explode("authorship.countries").alias("country_code")
    ) \
    .groupBy("country_code") \
    .agg(
        F.countDistinct("id").alias("paper_count_scores"),
        F.round(F.avg("mean_score"), 3).alias("avg_score")
    ) \
    .filter(F.col("paper_count_scores") >= 10)

# ── COMBINED TABLE — PARTS 2 & 3 JOINED ──────────────────────────────────────

# Left join so countries in Part 2 without score data still appear (null avg_score)
# Ordered by paper_count descending to match report table layout
print("Countries by matched paper count with acceptance rate and average reviewer score:")
q5_country_acceptance \
    .join(
        q5_country_scores.select("country_code", "avg_score"),
        on="country_code",
        how="left"
    ) \
    .join(country_mapping, on="country_code", how="left") \
    .withColumn("country_name",
        F.coalesce(F.col("country_name"), F.col("country_code"))
    ) \
    .withColumn("acceptance_rate_pct",
        F.round(F.col("acceptance_rate") * 100, 1)
    ) \
    .select("country_name", "paper_count", "acceptance_rate_pct", "avg_score") \
    .orderBy(F.col("paper_count").desc()) \
    .toDF("Country Name", "Number of Papers", "Acceptance Rate (%)", "Average Reviewer Score") \
    .show(10, truncate=False)

Matched papers with known decision: 1601
Part 1 — Top 10 countries by matched paper count:
+--------------+----------------+
|Country Name  |Number of Papers|
+--------------+----------------+
|United States |880             |
|United Kingdom|221             |
|China         |176             |
|Canada        |131             |
|Israel        |107             |
|Germany       |98              |
|Switzerland   |75              |
|France        |52              |
|South Korea   |38              |
|Japan         |37              |
+--------------+----------------+
only showing top 10 rows
Countries by matched paper count with acceptance rate and average reviewer score:
+--------------+----------------+-------------------+----------------------+
|Country Name  |Number of Papers|Acceptance Rate (%)|Average Reviewer Score|
+--------------+----------------+-------------------+----------------------+
|United States |880             |88.9               |6.281                 |
|United Kingdom|22

In [0]:
# ============================================================
# QUESTION 6 — LOW SCORE MATCHED PAPERS IN THE 95TH CITATION PERCENTILE
# ============================================================
# Uses ICLRdata filtered to non-empty scores. No decision required.
# mean_score < 5 defines a low score paper.
# citation_normalized_percentile.value >= 0.95 used as the pre-computed
# 95th percentile indicator from OpenAlex.

# Compute mean score per paper and filter to low-score papers
iclr_q6 = ICLRdata \
    .filter(F.size(F.col("scores")) > 0) \
    .withColumn("mean_score",
        F.aggregate(
            F.col("scores"), F.lit(0.0),
            lambda acc, x: acc + x.cast("double")
        ) / F.size(F.col("scores"))
    ) \
    .filter(F.col("mean_score") < 5)

print(f"Papers with mean score < 5: {iclr_q6.count()}")

# Prepare OpenAlex: extract title, citation count and normalised percentile value.
# Deduplicate by keeping the record with the highest percentile value per title.
# This ensures each title retains a single representative percentile record.
w_oa = Window.partitionBy("title")
openalex_q6 = openalex_raw \
    .select(
        F.col("title"),
        F.col("cited_by_count").cast("double"),
        F.col("citation_normalized_percentile.value").alias("percentile_value")
    ) \
    .withColumn("max_percentile", F.max("percentile_value").over(w_oa)) \
    .filter(F.col("percentile_value") == F.col("max_percentile")) \
    .drop("max_percentile") \
    .dropDuplicates(["title"]) \
    .withColumnRenamed("title", "oa_title")

print(f"OpenAlex records after dedup: {openalex_q6.count()}")

# Join low score ICLR papers to OpenAlex on exact title match
q6_joined = iclr_q6.join(
    openalex_q6,
    iclr_q6.title == openalex_q6.oa_title,
    how="inner"
).select(
    iclr_q6["id"],
    iclr_q6["title"],
    iclr_q6["year"],
    "mean_score", "cited_by_count", "percentile_value"
)

print(f"Low score papers with OpenAlex citation data: {q6_joined.count()}")

# Filter to papers in the 95th percentile using citation_normalized_percentile.value
q6_result = q6_joined \
    .filter(F.col("percentile_value") >= 0.95) \
    .select("id", "title", "year", "mean_score", "cited_by_count") \
    .orderBy(F.col("cited_by_count").desc())

print("Q6 Answer — low score papers in the 95th citation percentile:")
q6_result.toDF("ID", "Title", "ICLR Submission Year", "Average Score", "Cited By Count").show(truncate=False)
print(f"Total papers identified: {q6_result.count()}")

Papers with mean score < 5: 15045
OpenAlex records after dedup: 1219
Low score papers with OpenAlex citation data: 62
Q6 Answer — low score papers in the 95th citation percentile:
+----------+----------------------------------------------------------------------------------------+--------------------+------------------+--------------+
|ID        |Title                                                                                   |ICLR Submission Year|Average Score     |Cited By Count|
+----------+----------------------------------------------------------------------------------------+--------------------+------------------+--------------+
|S19eAF9ee |Structured Sequence Modeling with Graph Convolutional Recurrent Networks                |2017                |4.0               |763.0         |
|HkmaTz-0W |Visualizing the Loss Landscape of Neural Nets                                           |2018                |4.666666666666667 |514.0         |
|H1bM1fZCW |GradNorm: Gradient Norm

In [0]:
# ============================================================
# QUESTION 6 EXTENSION — CITATION IMPACT: ACCEPTED VS REJECTED
# LOW-SCORING PAPERS IN THE OPENALEX MATCHED SET
# ============================================================
# Among all 62 low-scoring papers with OpenAlex citation data,
# do accepted papers accumulate more citations than rejected ones?

# Join q6_joined (62 papers) to decisions from ICLRdata
q6_decisions = q6_joined.join(
    ICLRdata.select("id", "decision"),
    on="id",
    how="left"
).withColumn("outcome",
    F.when(
        F.col("decision").rlike("(?i)accept|oral|poster|spotlight|workshop|talk|notable"),
        "Accepted"
    ).otherwise("Rejected")
).filter(F.col("decision") != "")

print(f"Matched papers with a mean score < 5 and have a known decision: {q6_decisions.count()}")

q6_decisions.groupBy("outcome").agg(
    F.count("id").alias("papers"),
    F.round(F.avg("cited_by_count"), 1).alias("mean_citations"),
    F.round(F.min("cited_by_count"), 0).alias("min_citations"),
    F.round(F.max("cited_by_count"), 0).alias("max_citations")
).orderBy("outcome") \
    .toDF("Outcome", "Number of Papers", "Average Cited By Count", "Minimum Cited By Count", "Maximum Cited By Count") \
    .show(truncate=False)

Matched papers with a mean score < 5 and have a known decision: 62
+--------+----------------+----------------------+----------------------+----------------------+
|Outcome |Number of Papers|Average Cited By Count|Minimum Cited By Count|Maximum Cited By Count|
+--------+----------------+----------------------+----------------------+----------------------+
|Accepted|24              |40.3                  |0.0                   |514.0                 |
|Rejected|38              |55.6                  |0.0                   |763.0                 |
+--------+----------------+----------------------+----------------------+----------------------+

